In [4]:
from sklearn.metrics import roc_curve, roc_auc_score
import numpy as np
from data_loading.graphDataLoader import GraphDataLoader
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import re
import pickle
import os
import dcor

from utils.utils_training import output_directory
import utils.utils_evaluation as et

In [2]:
def make_dict(auc):
    """Create Dictionary with 

    Args:
        auc(dict): with keys SVJ_***_*p*

    Returns:  auc(dict) with the form: {(rinv, mMed): 0.6106412792103517, ...
        
    """
    dict_result = {}

    for key in auc.keys():
        pattern = r"SVJ_(\d+)_0p(\d+)"
        match = re.search(pattern, key)
        if match:
            mMed = int(match.group(1))
            rinv = float(f"0.{match.group(2)}")
            sig_key = (mMed, rinv)
            dict_result[sig_key] = auc[key]
    return dict_result

In [3]:
def plot_auc(auc, name, suffix = "_eval",  variable_name = "AUC"):
    """Plot AUC for all Mediator masses and rinv in one plot

    Args:
        auc(dict): as dict created in roc_auc()
        name(str): Name of the training for the saving of the pdf
        variable_name(str): Default AUC

    Returns:
        
    """

    auc = make_dict(auc)
        
    x_list = []
    y_list = []
    for key in auc.keys():
        x, y = key
        if x not in x_list: x_list.append(x)
        if y not in y_list: y_list.append(y)
    x_list = sorted(x_list)
    y_list = sorted(y_list)[::-1]  # ascending y axis

    gridxy = []
    for idx_y, y in enumerate(y_list):
        gridxy.append([])
        for idx_x, x in enumerate(x_list):
            key = (x, y)
            if key in auc.keys():
                value = auc[key]
            else:
                value = 0.
            gridxy[idx_y].append(value)

    # Create a copy of the 'viridis' colormap
    cmap = plt.get_cmap('viridis')

    # Create a new colormap that is the same as 'viridis', but with the color for the specific values set to white
    new_cmap = colors.LinearSegmentedColormap.from_list(
            'new_cmap', cmap(np.linspace(0, 1, cmap.N)), N=cmap.N)

    # Set the color for the specific values to white
    new_cmap.set_under('white', 0.01)

    # Draw AUC table using matplotlib
    plt.figure(figsize=(12, 12))
    im  = plt.imshow(gridxy, cmap=new_cmap, interpolation='nearest')
    # Show value auc in each bin
    for idx_x, x in enumerate(x_list):
        for idx_y, y in enumerate(y_list):
            plt.text(idx_x, idx_y, "{:.3f}".format(gridxy[idx_y][idx_x]), va='center', ha='center', color='white', fontsize=18)
  
    im.set_clim(0.5, 1.0)
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label(variable_name)
    
    plt.xticks(np.arange(len(x_list)), x_list)
    plt.yticks(np.arange(len(y_list)), y_list)
    plt.xlabel("$r_{inv}$")
    plt.ylabel("$M_{Z'}$")

    #hep.cms.label('Preliminary', data=False)
    #hep.cms.label(loc=0)

    output_path = output_directory(name)
    outout_plot_name = name + suffix + "_auc.png"
    print(f"{outout_plot_name} has been saved.")
    plt.savefig(os.path.join(output_path,outout_plot_name),  dpi=400)
    plt.close()
    
    return 

In [7]:
file_path = "/t3home/rhampp/SVJ_GNN/output/LundNet5+_SVJ-QCD-TTJet-EW_vs12/_eval_ROC_AUC.pkl"

In [8]:
with open(file_path, 'rb') as file:
    evaluation_dict = pickle.load(file) 
    print(f"Opened file: {file_path}")

Opened file: /t3home/rhampp/SVJ_GNN/output/LundNet5+_SVJ-QCD-TTJet-EW_vs12/_eval_ROC_AUC.pkl


In [13]:
auc = {}
for key in evaluation_dict:
    auc[key] = evaluation_dict[key]["auc"]

In [14]:
auc

{'SVJ_700_0p3_LN5p': 0.7317364312696509,
 'SVJ_700_0p5_LN5p': 0.7271524019091766,
 'SVJ_700_0p7_LN5p': 0.6938333731922189,
 'SVJ_800_0p3_LN5p': 0.8258008051581579,
 'SVJ_800_0p5_LN5p': 0.8081829167898014,
 'SVJ_800_0p7_LN5p': 0.76144566655842,
 'SVJ_900_0p3_LN5p': 0.8666168693893888,
 'SVJ_900_0p5_LN5p': 0.8330962806165834,
 'SVJ_900_0p7_LN5p': 0.7608691994907917,
 'SVJ_1000_0p3_LN5p': 0.8842360392245778,
 'SVJ_1000_0p5_LN5p': 0.8472170351894961,
 'SVJ_1000_0p7_LN5p': 0.7857296688046157,
 'SVJ_1100_0p3_LN5p': 0.8768184344800883,
 'SVJ_1100_0p5_LN5p': 0.8507440098622792,
 'SVJ_1100_0p7_LN5p': 0.8260332624690603,
 'SVJ_1200_0p3_LN5p': 0.8873869553794362,
 'SVJ_1200_0p5_LN5p': 0.8607583038467109,
 'SVJ_1200_0p7_LN5p': 0.8190478407599956,
 'SVJ_1300_0p3_LN5p': 0.8902923911411523,
 'SVJ_1300_0p5_LN5p': 0.8662484560326172,
 'SVJ_1300_0p7_LN5p': 0.8157314606677031,
 'SVJ_1400_0p3_LN5p': 0.8873846968102441,
 'SVJ_1400_0p5_LN5p': 0.8728578109610452,
 'SVJ_1400_0p7_LN5p': 0.8324487637840574,
 'S

In [9]:
name = "LundNet5+_SVJ-QCD-TTJet-EW_vs12"

In [15]:

et.plot_auc(auc, name)

LundNet5+_SVJ-QCD-TTJet-EW_vs12_eval_auc.png has been saved.
